In [5]:
import numpy as np
import torch
import pandas as pd
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [6]:
# 数据集下载
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [10]:
class MyCNN(torch.nn.Module):
    def __init__(self, input_channel, output_channel, num_classes):
        super(MyCNN, self).__init__()

        self.in_layer = self.Block(input_channel, 16)
        self.hide_layer = self.Block(16, 32)
        self.out_layer = self.Block(32, output_channel)

        self.classifier = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Linear(output_channel, out_features=num_classes)
        )
    
    def Block(self, input_channel, output_channel):
        sequential = torch.nn.Sequential(
        torch.nn.Conv2d(in_channels=input_channel, out_channels=output_channel, kernel_size=3, stride=1, padding=1, bias=False),
        torch.nn.BatchNorm2d(output_channel),
        torch.nn.ReLU(inplace=True)
        )
        return sequential

    def forward(self, x):
        x = self.in_layer(x)
        x = self.hide_layer(x)
        x = self.out_layer(x)
        x = torch.nn.Flatten(x, start_dim=1)
        x = self.classfier(x)
        return x

In [ ]:
epochs = 20
batch_size = 128
learning_rate = 0.01
num_classes = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MyCNN(3, 64, num_classes).to(device)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.3)
softmax = torch.nn.Softmax(dim=1)


In [13]:
for epoch in range(epochs):
    model.train()
    for batch, (img, label) in enumerate(train_dataloader):
        img = img.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        forward_result = model(img)
        loss = criterion(forward_result, label)
        loss.backward()
        optimizer.step()

        output = softmax(output)
        pred = output.argmax(dim=1)

    model.eval()
    with torch.no_grad():
        for batch, (img, label) in enumerate(test_dataloader):
            img = img.to(device)
            label = label.to(device)

            forward_result = model(img)
            output = softmax(forward_result)
            pred = output.argmax(dim=1)


RuntimeError: Given groups=1, weight of size [16, 3, 3, 3], expected input[128, 1, 28, 28] to have 3 channels, but got 1 channels instead